In [ ]:
class Initialization:
    """
    Class 1: Handles plant parameter storage, physical ODE evaluations, 
    and simulates the real-time sensor streaming phase under input dither excitation.
    """
    def __init__(self):
        # 1. Hardware Sensor Setup
        self.dt = 0.01          # 100 Hz hardware sensor sampling frequency
        self.n_samples = 1000   # Number of continuous snapshots collected
        
        # 2. Plant Physical Ground-Truth Constants
        self.q_V = 1.0          # Volumetric space velocity (q/V) [s^-1]
        self.C_Af = 1.0         # Feed concentration of reactant A [kmol/m^3]
        self.T_f = 350.0        # Feed temperature [K]
        self.k_0 = 1e8          # Arrhenius pre-exponential kinetic constant [s^-1]
        self.E_R = 6000.0       # Activation energy over gas constant (E/R) [K]
        self.dH_term = 2e5      # Dimensionless adiabatic heat of reaction term [K*m^3/kmol]
        self.UA_term = 0.5      # Jacket heat transfer coefficient term [s^-1]
        
        # 3. Unstable Steady-State Target Equilibrium (The Local Origin)
        self.C_A_ss = 0.5       # Steady-state concentration [kmol/m^3]
        self.T_ss = 400.0       # Steady-state temperature [K]
        self.T_c_ss = 350.0     # Steady-state nominal coolant jacket temperature [K]

    def cstr_nonlinear_dynamics(self, C_A, T, T_c):
        """
        Evaluates the exact, non-linear physical ordinary differential equations 
        governing the internal material and energy balances of the reactor.
        """
        # Arrhenius rate law expression
        reaction_rate = self.k_0 * np.exp(-self.E_R / T) * C_A
        
        # Mass Balance: d(C_A)/dt
        dC_A = self.q_V * (self.C_Af - C_A) - reaction_rate
        
        # Energy Balance: d(T)/dt
        dT = self.q_V * (self.T_f - T) + self.dH_term * reaction_rate - self.UA_term * (T - T_c)
        
        return np.array([dC_A, dT])

    def stream_sensor_data(self):
        """
        Simulates an active plant dataset by generating a stream of sensor states
        paired with exact analytical continuous-time vector field snapshots.
        """
        np.random.seed(42)  # Seed generator for reproducible excitation patterns
        
        # Displace initial states slightly off-equilibrium to stimulate dynamics
        C_A = 0.55
        T = 405.0
        
        X_deviations = []
        U_deviations = []
        X_dot_analytical = []
        
        for k in range(self.n_samples):
            # Industrial Actuator Excitation: Sinusoidal dither with noise 
            # designed to sweep the vector field and expose structural stretching.
            T_c = self.T_c_ss + np.sin(k * 0.05) * 8.0 + np.random.normal(0, 0.2)
            
            # Capture the exact analytical derivative vector directly from the ODEs
            x_dot = self.cstr_nonlinear_dynamics(C_A, T, T_c)
            
            # Document and append localized state and control deviations from equilibrium
            X_deviations.append([C_A - self.C_A_ss, T - self.T_ss])
            U_deviations.append([T_c - self.T_c_ss])
            X_dot_analytical.append(x_dot)
            
            # March the plant state forward in continuous time
            C_A += x_dot * self.dt
            T += x_dot * self.dt

        return (np.array(X_deviations).T, 
                np.array(U_deviations).T, 
                np.array(X_dot_analytical).T)